# 35. SOLID refactoring and method dispatch

**Level:** Intermediate to advanced

**Learning outcome:** Refactor one mixed responsibility into small contracts and understand Python's real dispatch choices.

**Prerequisites:** 04-10 and 16-19: collaborators, contracts, patterns, and tests.

[Learning-path index](00_START_HERE_OOP_Learning_Path.ipynb) | [Previous](34_Weak_References_and_Object_Lifetime.ipynb)

## How to study this notebook
1. Read the mental model and explanation before executing a code cell.
2. Predict the output, then run the cell and compare the stored result.
3. Trace constructor calls, attribute reads/writes, returned values, and exception paths.
4. Change one input, predict the difference, and keep meaningful assertions.
5. Complete the practice prompts and answer the final checkpoint aloud.

**Run independently:** every worked code cell includes its own imports, definitions, and input data. You do not need to run another notebook first. Use Python 3.11+ and a Python Jupyter kernel. Only the standard library is used. Async cells use top-level await, not asyncio.run inside Jupyter.

**Offline safety:** all network states, commands, approvals, and failures are simulations. The code does not connect to devices, use credentials, write device configuration, or need external input files. Teaching limits and command strings are not vendor specifications. Assertions are learning checks, not production input validation.

## Mental model
SOLID is a set of design questions, not a required class count. Single responsibility separates
reasons to change. Open/closed asks whether a new behavior can be supplied without rewriting a
stable client. Liskov substitution asks whether a replacement honors its promised inputs, outputs,
side effects, and failures. Interface segregation keeps clients dependent only on the operations
they need. Dependency inversion makes orchestration depend on a contract rather than a particular
transport library.

Python supports overriding at runtime, defaults and keyword arguments for flexible signatures,
and explicit dispatch mechanisms. It does not keep several ordinary methods with the same name
merely because they have different parameter counts: a later definition replaces the earlier
binding. typing.overload describes signatures to type checkers but still requires one runtime
implementation. singledispatchmethod is a different mechanism: it dispatches on the first
non-self/non-cls argument's runtime type.

Refactoring should improve an actual seam. Keep parsing, policy, storage, and formatting separate
when they change independently, but do not build a generic dependency container for three simple
objects. Verify behavior before and after the refactor. Prefer composition over a subclass that
silently strengthens a base precondition or changes a read into a write.

## Worked example 1: Split source, policy, and formatting responsibilities

**Problem:** Evaluate the same collected observation with interchangeable policies without touching collection code.

### Step-by-step explanation
1. Source owns collection only and returns a tiny offline record.
2. MaximumLoad owns the decision and its threshold.
3. Reporter composes both rather than subclassing every threshold variant.
4. Check two policies against the same source to demonstrate the extension boundary.

**Expected output:**
```text
same source, different policy, unchanged orchestration
```

**Common mistake / limitation:** This is a small illustration, not a complete interface schema or transport error strategy.

**Verification:** the next code cell asserts results, state, or expected rejection. Compare those checks with the explanation, not just the printed text.

In [1]:
class Source:
    def read(self):
        return {"hostname": "edge-01", "load": 75}

class MaximumLoad:
    def __init__(self, limit):
        self.limit = limit

    def accepts(self, observation):
        return observation["load"] <= self.limit

class Reporter:
    def __init__(self, source, policy):
        self.source, self.policy = source, policy

    def report(self):
        observation = self.source.read()
        state = "pass" if self.policy.accepts(observation) else "fail"
        return f"{observation['hostname']}:{state}"

source = Source()
assert Reporter(source, MaximumLoad(80)).report() == "edge-01:pass"
assert Reporter(source, MaximumLoad(70)).report() == "edge-01:fail"
print("same source, different policy, unchanged orchestration")

same source, different policy, unchanged orchestration


### Practice 1

Inject a formatter as a third collaborator and verify the collected record does not change.

Predict the result before executing your extension. Keep one normal case and one edge/rejection case. State whether the extension changes identity, ownership, input validation, or error propagation.

## Worked example 2: Defaults replace imagined signature overloading

**Problem:** Use one clear runtime implementation for optional behavior and reject accidental positional ambiguity.

### Step-by-step explanation
1. Define one method instead of writing two same-named method bodies.
2. Use the default for the common brief form.
3. Make detail keyword-only so calls express their intent.
4. Prove a second positional value is not accepted by the public signature.

**Expected output:**
```text
ambiguous positional option rejected
{'hostname': 'edge-01', 'mode': 'detail'}
```

**Common mistake / limitation:** A type checker overload declaration would describe accepted signatures, not create extra runtime implementations.

**Verification:** the next code cell asserts results, state, or expected rejection. Compare those checks with the explanation, not just the printed text.

In [2]:
class StatusReader:
    def read(self, hostname, *, detail=False):
        return {"hostname": hostname, "mode": "detail" if detail else "brief"}

reader = StatusReader()
assert reader.read("edge-01") == {"hostname": "edge-01", "mode": "brief"}
assert reader.read("edge-01", detail=True)["mode"] == "detail"
try:
    reader.read("edge-01", True)
except TypeError:
    print("ambiguous positional option rejected")
else:
    raise AssertionError("keyword-only contract bypassed")
print(reader.read("edge-01", detail=True))

ambiguous positional option rejected
{'hostname': 'edge-01', 'mode': 'detail'}


### Practice 2

Add a keyword-only include_counters option and define its output contract.

Predict the result before executing your extension. Keep one normal case and one edge/rejection case. State whether the extension changes identity, ownership, input validation, or error propagation.

## Worked example 3: Explicit runtime type dispatch with singledispatchmethod

**Problem:** Render a domain value and plain text through registered runtime implementations.

### Step-by-step explanation
1. Use singledispatchmethod for one explicit runtime dispatch contract.
2. Register implementations whose first non-self argument is annotated with a concrete type.
3. Call one render entry point with two supported value types.
4. Keep the generic fallback an explicit error for unsupported types.

**Expected output:**
```text
unsupported value type: int
device:edge-01
```

**Common mistake / limitation:** Dispatch uses runtime type relationships, not arbitrary predicates or every parameter in a signature.

**Verification:** the next code cell asserts results, state, or expected rejection. Compare those checks with the explanation, not just the printed text.

In [3]:
from dataclasses import dataclass
from functools import singledispatchmethod

@dataclass(frozen=True)
class Device:
    hostname: str

class Renderer:
    @singledispatchmethod
    def render(self, value):
        raise TypeError(f"unsupported value type: {type(value).__name__}")

    @render.register
    def render_device(self, value: Device):
        return "device:" + value.hostname

    @render.register
    def render_text(self, value: str):
        return "text:" + value

renderer = Renderer()
assert renderer.render(Device("edge-01")) == "device:edge-01"
assert renderer.render("lab") == "text:lab"
try:
    renderer.render(42)
except TypeError as error:
    assert "int" in str(error)
    print(str(error))
else:
    raise AssertionError("unsupported type silently rendered")
print(renderer.render(Device("edge-01")))

unsupported value type: int
device:edge-01


### Practice 3

Add a structured Interface value and register a renderer without editing the existing two implementations.

Predict the result before executing your extension. Keep one normal case and one edge/rejection case. State whether the extension changes identity, ownership, input validation, or error propagation.

## Check your understanding: SOLID refactoring and method dispatch

**You should now be able to:** Refactor one mixed responsibility into small contracts and understand Python's real dispatch choices.

1. Explain which object owns each piece of state and which operations can change it.
2. Walk through a normal result and an invalid-input or failure path without running the code.
3. Name one invariant, one useful assertion, and one production assumption not proved by the fake.
4. Implement one extension prompt, including a test that would fail if your new behavior were wrong.
5. Explain when a function or a dataclass would be simpler than another subclass or framework.

**Do not confuse:** a successful local assertion with real device compatibility; missing data with a healthy result; underscore naming with access control; type annotations with runtime validation; or local rollback with a multi-device transaction.

## Official reference reading
- `https://docs.python.org/3.13/library/functools.html`
- `https://docs.python.org/3.13/library/typing.html`
- `https://docs.python.org/3.13/tutorial/classes.html`

References describe Python contracts. They do not certify any vendor command or network policy.